In [241]:
import numpy as np
import pandas as pd

from math import log10 as lg
from math import comb

import seaborn as sns
from scipy.linalg import svd

import matplotlib.pyplot as plt

In [242]:
factors = 7
n_candidates = 2 ** factors  
n_experiments = 24

In [243]:
# 生成所有可能的实验点（候选集）
levels = np.array([-1, 1])
candidate_set = np.array(np.meshgrid(*[levels]*factors, indexing='ij')).T.reshape(-1, factors)

In [244]:
# 编辑关心的效应

# 考虑主效应、二阶交互效应
complete_factors = 2

# 不考虑主效应，只考虑与完全因子的二阶交互效应
semi_factors = 1

# 只考虑主效应，不考虑与完全因子的二阶交互效应
main_factors = 0

# 其余：考虑主效应，只考虑与完全因子的二阶交互效应

# complete_factors,other,semi_factors,main_factors

In [245]:
def effects_X(design):
    interactions = []
    
    for k in range(complete_factors):
        for i in range(k+1,factors-main_factors):
            interaction = design[:, k:k+1] * design[:, i:i+1]  # 逐元素相乘
            interactions.append(interaction)

    # 将所有交互效应合并为一个矩阵
    interaction_matrix = np.hstack(interactions)  if interactions else np.zeros((n_experiments, 0))
    
    # 合并主效应和交互效应
    X = np.hstack([
        np.ones((n_experiments, 1)),
        design[:, :factors-semi_factors-main_factors], 
        design[:, factors-main_factors:], 
        interaction_matrix
    ])
    
    return X

# D op
# O(n_effects^2 * n_experiments)
def object_func(design):
    X = effects_X(design)
    M = X.T @ X 
    # M_inv = np.linalg.inv(M)
    return np.linalg.det(M)

def rate_object_func(design):
    X = effects_X(design)    
    M = X.T @ X 
    # M_inv = np.linalg.inv(M)
    n_effects = len(X[0])
    # print(n_effects)
    return np.linalg.det(M)/ n_experiments**n_effects

# O(n_experiments^4 * 2^factors)
def d_optimal_design_exchange(X, n_experiments, max_iter=100, early_stop = True ,tol= 1e-8):
    """
        使用交换算法求解D-最优设计
        
        参数:
        X (np.ndarray): 候选集矩阵，每行代表一个实验点
        n_experiments (int): 需要选择的实验次数
        max_iter (int): 最大迭代次数
        tol (float): 收敛容差
        
        返回:
        np.ndarray: 选中的实验点索引
    """
    n_candidates, n_factors = X.shape
    
    # 初始化：随机选择n_experiments个实验点
    selected_indices = np.random.choice(n_candidates, n_experiments, replace=False)
    current_design = X[selected_indices]
    
    # 计算初始行列式
    current_det = object_func(current_design)
    
    for iteration in range(max_iter):
        improved = False
        best_det = current_det
        best_swap = None
        
        # 尝试交换每个选中的点与每个未选中的点
        for i, selected_idx in enumerate(selected_indices):
            for candidate_idx in range(n_candidates):
                if candidate_idx not in selected_indices:
                    # 临时交换
                    temp_indices = selected_indices.copy()
                    temp_indices[i] = candidate_idx
                    temp_design = X[temp_indices]
                    
                    # 计算新的行列式
                    temp_det = object_func(temp_design) 
                    
                    # 如果新行列式更大，记录这个交换
                    if temp_det > best_det:
                        best_det = temp_det
                        best_swap = (i, candidate_idx)
        
        # 执行最佳交换（如果有改进）
        if best_swap is not None and best_det > current_det * (1 + tol):
            i, candidate_idx = best_swap
            selected_indices[i] = candidate_idx
            current_det = best_det
            improved = True
        
        # 如果没有改进，算法收敛
        if early_stop:
            if not improved:    
                print('early stop')
                break
        
    return selected_indices


In [246]:
selected_indices = d_optimal_design_exchange(candidate_set, n_experiments, max_iter = 1000)
# selected_indices = d_optimal_design_ga(candidate_set, n_experiments, 1000)

early stop


In [247]:
# selected_indices = [ 12,  27,  29,  41,  54,  65,  70,  88, 100, 106, 119, 133, 142,148, 160, 175, 177, 195, 210, 223, 252, 253]

In [248]:

selected_indices.sort()
optimal_design = candidate_set[selected_indices]

# 输出结果
print(f"candidates size: {n_candidates}")
print(f"experiments size: {n_experiments}")
print(f"selected indices: {selected_indices}")

print(f"vertical rate: {rate_object_func(optimal_design)}")


candidates size: 128
experiments size: 24
selected indices: [  5   9  20  23  24  30  32  33  38  39  58  59  60  61  66  75  76  81
 109 110 112 115 118 127]
vertical rate: 0.25856350617532725


In [249]:
print(f"optimal design:\n{optimal_design}")


optimal design:
[[ 1 -1  1 -1 -1 -1 -1]
 [ 1 -1 -1  1 -1 -1 -1]
 [-1 -1  1 -1  1 -1 -1]
 [ 1  1  1 -1  1 -1 -1]
 [-1 -1 -1  1  1 -1 -1]
 [-1  1  1  1  1 -1 -1]
 [-1 -1 -1 -1 -1  1 -1]
 [ 1 -1 -1 -1 -1  1 -1]
 [-1  1  1 -1 -1  1 -1]
 [ 1  1  1 -1 -1  1 -1]
 [-1  1 -1  1  1  1 -1]
 [ 1  1 -1  1  1  1 -1]
 [-1 -1  1  1  1  1 -1]
 [ 1 -1  1  1  1  1 -1]
 [-1  1 -1 -1 -1 -1  1]
 [ 1  1 -1  1 -1 -1  1]
 [-1 -1  1  1 -1 -1  1]
 [ 1 -1 -1 -1  1 -1  1]
 [ 1 -1  1  1 -1  1  1]
 [-1  1  1  1 -1  1  1]
 [-1 -1 -1 -1  1  1  1]
 [ 1  1 -1 -1  1  1  1]
 [-1  1  1 -1  1  1  1]
 [ 1  1  1  1  1  1  1]]


In [250]:
pd.DataFrame(optimal_design).to_excel(
    'optimal_design.xlsx', 
    index_label='Trial', 
    header=[chr(65 + i) for i in range(factors)]
)